## Limpieza y procesamiento de datos para Datos Biométricos de un SmartWatch

### Integrantes:
- VINCENT FARENDEN CERON
- RODRIGO IGNACIO MARTINEZ BECKER
- DIEGO IGNACIO PENA Y LILLO LUHRS

### Asignatura:
- MACHINE LEARNING_001D

### Profesor:
- FRANCISCO JAVIER JEREZ SALAZAR

### Fecha:
- 13-09-2026

---

## Introducción y objetivo del informe

El presente informe técnico documenta el proceso de limpieza y preparación de un
conjunto de datos biométricos provenientes de un smartwatch. El objetivo es dejar
los datos correctamente representados y listos para su posterior uso en modelos de
Machine Learning.

El trabajo se desarrolla en Python, utilizando bibliotecas como **pandas**, **NumPy**,
**Matplotlib/Seaborn** y **Scikit-learn**, y se organiza en los siguientes apartados:

- Descripción general del conjunto de datos.
- Selección de características relevantes.
- Detección y gestión de valores faltantes y atípicos.
- Análisis de correlación entre variables.
- Escalamiento y codificación de variables.
- Conclusiones.

---

# Descripción general del conjunto de datos

Este dataset simula los registros biométricos capturados por un smartwatch. Cada fila
corresponde a una medición asociada a un usuario e incluye las siguientes variables:

| Variable | Descripción |
| --- | --- |
| ID de usuario | Identificador del portador del dispositivo. |
| Pulsaciones por minuto (BPM) | Frecuencia cardíaca registrada. |
| Nivel de oxígeno en sangre (%) | Saturación de oxígeno. |
| Contador de pasos | Cantidad de pasos registrados. |
| Duración del sueño (horas) | Horas dormidas. |
| Nivel de actividad | Categoría de actividad física (variable categórica). |
| Nivel de estrés | Indicador del estado de estrés. |

Los datos no están normalizados y presentan valores nulos, inconsistencias de
formato y valores atípicos, lo que justifica el proceso de limpieza descrito en este
informe.

### Carga del conjunto de datos

In [1]:
import pandas as pd
import numpy as np

RUTA_DATOS = "../data/raw/unclean_smartwatch_health_data.csv"

df = pd.read_csv(RUTA_DATOS)

df.head()

,User ID,Heart Rate (BPM),Blood Oxygen Level (%),Step Count,Sleep Duration (hours),Activity Level,Stress Level
0,4174.0,58.939776,98.809650,5450.390578,7.167235622316564,Highly Active,1
1,NaN,NaN,98.532195,727.601610,6.538239375570314,Highly_Active,5
2,1860.0,247.803052,97.052954,2826.521994,ERROR,Highly Active,5
3,2294.0,40.000000,96.894213,13797.338044,7.367789630207228,Actve,3
4,2130.0,61.950165,98.583797,15679.067648,NaN,Highly_Active,6


### Dimensiones, tipos de datos y estadísticos descriptivos

Antes de limpiar es necesario conocer el tamaño del dataset, el tipo de dato con el que
pandas interpretó cada columna y el comportamiento general de las variables numéricas.

In [17]:
filas, columnas = df.shape

print(f"""El presente dataset tiene {filas} registros y {columnas} columnas.

Columnas disponibles:
{chr(10).join("  - " + c for c in df.columns)}
""")

El presente dataset tiene 10000 registros y 7 columnas.

Columnas disponibles:
  - User ID
  - Heart Rate (BPM)
  - Blood Oxygen Level (%)
  - Step Count
  - Sleep Duration (hours)
  - Activity Level
  - Stress Level



#### Tipos de datos y valores faltantes

Se revisa el tipo asignado a cada columna junto con la cantidad de valores nulos, para
detectar variables que quedaron mal tipificadas.

In [18]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(),
})

resumen

,tipo,no_nulos,nulos,% nulos,valores_unicos
User ID,float64,9799,201,2.01,3634
Heart Rate (BPM),float64,9600,400,4.00,9517
Blood Oxygen Level (%),float64,9700,300,3.00,8175
Step Count,float64,9900,100,1.00,9900
Sleep Duration (hours),str,9850,150,1.50,9604
Activity Level,str,9800,200,2.00,6
Stress Level,str,9800,200,2.00,11


#### Estadísticos descriptivos

Se calculan los estadísticos de las variables numéricas (media, desviación estándar,
mínimo, cuartiles y máximo) y, por separado, el resumen de las variables de tipo texto.

In [16]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
User ID,9799.0,3007.48,1150.58,1001.00,1997.50,2998.00,4004.00,4999.00
Heart Rate (BPM),9600.0,76.04,19.41,40.00,64.89,75.22,85.20,296.59
Blood Oxygen Level (%),9700.0,97.84,1.73,90.79,96.66,98.01,99.38,100.00
Step Count,9900.0,6985.69,6885.81,0.91,2021.04,4962.53,9724.90,62486.69


In [15]:
df.describe(include="str").T

,count,unique,top,freq
Sleep Duration (hours),9850,9604,ERROR,247
Activity Level,9800,6,Seddentary,1676
Stress Level,9800,11,2,1007


#### Categorías presentes en las variables de texto

Se listan los valores únicos de las columnas no numéricas para identificar
inconsistencias de escritura y valores que no corresponden a la variable.

In [19]:
def es_numero(valor):
    """Indica si un texto puede convertirse a numero."""
    try:
        float(valor)
        return True
    except ValueError:
        return False


for columna in df.select_dtypes(include="str").columns:
    valores = sorted(df[columna].dropna().unique().tolist())
    no_numericos = [v for v in valores if not es_numero(v)]

    if len(valores) <= 15:
        print(f"{columna} -> {len(valores)} categorias: {valores}")
    else:
        print(f"{columna} -> {len(valores)} valores unicos (mayormente numericos)")
        print(f"   texto no numerico encontrado: {no_numericos}")
    print()

Sleep Duration (hours) -> 9604 valores unicos (mayormente numericos)
   texto no numerico encontrado: ['ERROR']

Activity Level -> 6 categorias: ['Active', 'Actve', 'Highly Active', 'Highly_Active', 'Seddentary', 'Sedentary']

Stress Level -> 11 categorias: ['1', '10', '2', '3', '4', '5', '6', '7', '8', '9', 'Very High']



---

# Selección de características relevantes

_En este apartado se identifican las variables que aportan información útil para el
análisis y se descartan aquellas que no lo hacen (por ejemplo, identificadores sin
valor predictivo o variables con varianza nula)._

### Criterios de selección

_Se definen los criterios aplicados: relevancia respecto al fenómeno estudiado,
proporción de valores faltantes, variabilidad y redundancia entre variables._

### Características seleccionadas y descartadas

_Se listan las variables retenidas y las eliminadas, justificando cada decisión._

---

# Detección y gestión de valores faltantes y atípicos

### Valores faltantes

_Cuantificación de los valores nulos por variable y análisis de su distribución._

### Estrategia de imputación o eliminación

_Se define y justifica la técnica aplicada a cada variable: eliminación de registros,
imputación por media/mediana/moda u otra estrategia estadística._

### Valores atípicos

_Detección de outliers mediante técnicas estadísticas (rango intercuartílico, z-score)
y apoyo visual con boxplots e histogramas._

### Tratamiento de los valores atípicos

_Decisión sobre cada caso detectado: corrección, imputación, acotación o eliminación,
indicando el criterio utilizado._

---

# Análisis de correlación entre variables

### Matriz de correlación

_Cálculo de la correlación entre las variables numéricas y su representación mediante
un mapa de calor._

### Interpretación de las correlaciones

_Análisis de las relaciones detectadas: variables fuertemente correlacionadas entre sí
y posible redundancia de información._

### Eliminación o retención de características

_Decisión final sobre qué variables se mantienen y cuáles se eliminan a partir del
análisis de correlación, con su justificación._

---

# Escalamiento y codificación de variables

### Codificación de variables categóricas

_Transformación de las variables categóricas (por ejemplo, el nivel de actividad) a
formato numérico mediante Label Encoding u One-Hot Encoding, según corresponda._

### Escalamiento de variables numéricas

_Aplicación de una técnica de escalamiento (normalización Min-Max o estandarización
Z-score) a las variables numéricas, justificando la elección._

### Dataset final

_Presentación del conjunto de datos resultante, ya limpio, codificado y escalado._

---

# Conclusiones

_Síntesis del proceso realizado: principales problemas detectados en los datos,
decisiones de limpieza adoptadas y su justificación, características finales
seleccionadas y estado del dataset para su uso en etapas posteriores de modelado._